In [3]:
import os
print(os.getcwd())
# copy that output and use it in the set_tracking_uri above
# ```

# So if `os.getcwd()` gives you `/Users/kousthubhngowda/Documents/External/Fraud Detection Model/fraud-detection-system`, your URI becomes:
# ```
# file:///Users/kousthubhngowda/Documents/External/Fraud Detection Model/fraud-detection-system/mlruns

/Users/kousthubhngowda/Documents/External/Fraud Detection Model/fraud-detection-system/notebooks


In [13]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import numpy as np
import pandas as pd
import xgboost as xgb
import json
from pathlib import Path
from sklearn.metrics import average_precision_score, roc_auc_score
import os

# Set tracking to local mlruns folder inside your project
os.chdir("/Users/kousthubhngowda/Documents/External/Fraud Detection Model/fraud-detection-system")
mlflow.set_tracking_uri("mlruns")
mlflow.set_experiment("fraud-detection")

print(f"MLflow version: {mlflow.__version__}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Working dir: {os.getcwd()}")
print("Setup complete!")

MLflow version: 3.10.1
Tracking URI: mlruns
Working dir: /Users/kousthubhngowda/Documents/External/Fraud Detection Model/fraud-detection-system
Setup complete!


In [16]:
DATA_DIR = Path("data/processed")

X_train = np.load(DATA_DIR / "X_train_sm.npy")
y_train = np.load(DATA_DIR / "y_train_sm.npy")
X_test  = np.load(DATA_DIR / "X_test_scaled.npy")
y_test  = np.load(DATA_DIR / "y_test.npy")
feature_names = pd.read_csv(DATA_DIR / "feature_names.csv").iloc[:, 0].tolist()

best_params  = json.load(open("models/best_params.json"))
metrics_prev = json.load(open("models/metrics.json"))

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Features: {len(feature_names)}")
print(f"\nBest params from Week 3:\n{json.dumps(best_params, indent=2)}")
print(f"\nWeek 3 metrics:\n{json.dumps(metrics_prev, indent=2)}")

Train: (250196, 36) | Test: (56962, 36)
Features: 36

Best params from Week 3:
{
  "n_estimators": 635,
  "max_depth": 5,
  "learning_rate": 0.08414692163102593,
  "subsample": 0.8537878056349802,
  "colsample_bytree": 0.602687133118449,
  "min_child_weight": 1,
  "gamma": 0.25292708493449423,
  "reg_alpha": 0.005386210542316519,
  "reg_lambda": 1.9632228215437328,
  "scale_pos_weight": 199.27144351846263,
  "eval_metric": "aucpr",
  "random_state": 42,
  "verbosity": 0,
  "n_jobs": -1
}

Week 3 metrics:
{
  "auc_roc": 0.9803,
  "auc_pr": 0.8814,
  "precision": 0.7727,
  "recall": 0.8673,
  "f1": 0.8173,
  "true_positives": 85,
  "false_positives": 25,
  "false_negatives": 13,
  "true_negatives": 56839,
  "fraud_caught_pct": 86.73
}


In [17]:
from sklearn.linear_model import LogisticRegression

with mlflow.start_run(run_name="baseline_logistic_regression"):
    mlflow.set_tag("model_type", "baseline")
    mlflow.set_tag("dataset",    "creditcard-kaggle")

    mlflow.log_param("model",        "LogisticRegression")
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("train_size",   X_train.shape[0])
    mlflow.log_param("n_features",   X_train.shape[1])

    lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
    lr.fit(X_train, y_train)

    y_proba = lr.predict_proba(X_test)[:, 1]
    auc_roc = roc_auc_score(y_test, y_proba)
    auc_pr  = average_precision_score(y_test, y_proba)

    mlflow.log_metric("auc_roc", round(auc_roc, 4))
    mlflow.log_metric("auc_pr",  round(auc_pr, 4))
    mlflow.sklearn.log_model(lr, artifact_path="model")

    print(f"Baseline logged")
    print(f"AUC-ROC: {auc_roc:.4f} | AUC-PR: {auc_pr:.4f}")
    print(f"Run ID: {mlflow.active_run().info.run_id}")

2026/03/15 13:44:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/15 13:44:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Baseline logged
AUC-ROC: 0.9682 | AUC-PR: 0.6979
Run ID: 4304181e281c40cb8dd6b737d505cd6d


In [18]:
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score

with mlflow.start_run(run_name="xgboost_optuna_tuned") as run:
    mlflow.set_tag("model_type",          "champion")
    mlflow.set_tag("dataset",             "creditcard-kaggle")
    mlflow.set_tag("feature_engineering", "smote+robustscaler+interactions")

    # Log hyperparameters
    mlflow.log_params({k: v for k, v in best_params.items()
                       if k not in ("eval_metric", "verbosity", "n_jobs")})

    # Log dataset stats
    mlflow.log_param("train_size",      X_train.shape[0])
    mlflow.log_param("test_size",       X_test.shape[0])
    mlflow.log_param("n_features",      X_train.shape[1])
    mlflow.log_param("smote_strategy",  0.1)
    mlflow.log_param("train_fraud_pct", round(y_train.mean()*100, 2))
    mlflow.log_param("test_fraud_pct",  round(y_test.mean()*100, 4))

    # Train
    model = xgb.XGBClassifier(**best_params)
    model.fit(X_train, y_train)

    # Metrics
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred  = (y_proba >= 0.5).astype(int)
    cm      = confusion_matrix(y_test, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    mlflow.log_metrics({
        "auc_roc":          round(roc_auc_score(y_test, y_proba), 4),
        "auc_pr":           round(average_precision_score(y_test, y_proba), 4),
        "precision":        round(precision_score(y_test, y_pred, zero_division=0), 4),
        "recall":           round(recall_score(y_test, y_pred, zero_division=0), 4),
        "f1":               round(f1_score(y_test, y_pred, zero_division=0), 4),
        "true_positives":   int(tp),
        "false_positives":  int(fp),
        "false_negatives":  int(fn),
        "true_negatives":   int(tn),
        "fraud_caught_pct": round(tp / (tp + fn + 1e-9) * 100, 2),
    })

    # Feature importance artifact
    importance_df = pd.DataFrame({
        "feature":    feature_names,
        "importance": model.feature_importances_
    }).sort_values("importance", ascending=False)
    importance_df.to_csv("models/feature_importance.csv", index=False)
    mlflow.log_artifact("models/feature_importance.csv")

    # Register model in MLflow Model Registry
    mlflow.xgboost.log_model(
        model,
        artifact_path="model",
        registered_model_name="fraud_detector",
    )

    champion_run_id = run.info.run_id
    print(f"Champion run logged!")
    print(f"Run ID: {champion_run_id}")
    print(f"AUC-PR:  {round(average_precision_score(y_test, y_proba), 4)}")
    print(f"AUC-ROC: {round(roc_auc_score(y_test, y_proba), 4)}")

2026/03/15 13:44:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Champion run logged!
Run ID: 37c62dd6ec6e4329b92eebf481515007
AUC-PR:  0.8814
AUC-ROC: 0.9803


/Users/kousthubhngowda/Documents/External/Fraud Detection Model/.venv/lib/python3.12/site-packages/mlflow/tracking/_model_registry/utils.py:220: FutureWarning: The filesystem model registry backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri)
Successfully registered model 'fraud_detector'.
Created version '1' of model 'fraud_detector'.


In [19]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Get latest registered version
versions = client.get_latest_versions("fraud_detector")
latest   = versions[-1]

print(f"Model:   {latest.name}")
print(f"Version: {latest.version}")
print(f"Stage:   {latest.current_stage}")

# Promote to Production
client.transition_model_version_stage(
    name="fraud_detector",
    version=latest.version,
    stage="Production",
    archive_existing_versions=True,
)
print(f"\nVersion {latest.version} promoted to Production!")

Model:   fraud_detector
Version: 1
Stage:   None

Version 1 promoted to Production!


/var/folders/n_/g54j8x895rl59szwgfg17mf00000gn/T/ipykernel_81211/3210764011.py:6: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  versions = client.get_latest_versions("fraud_detector")
/var/folders/n_/g54j8x895rl59szwgfg17mf00000gn/T/ipykernel_81211/3210764011.py:14: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


In [20]:
production_model = mlflow.xgboost.load_model("models:/fraud_detector/Production")

test_sample = X_test[:5]
preds       = production_model.predict_proba(test_sample)[:, 1]

print("Production model loaded successfully!")
print(f"Sample fraud probabilities: {preds.round(4)}")
print(f"True labels:                {y_test[:5].astype(int)}")

Production model loaded successfully!
Sample fraud probabilities: [0.     0.     0.     0.     0.0001]
True labels:                [0 0 0 0 0]


In [21]:
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]

for t in thresholds:
    with mlflow.start_run(run_name=f"xgboost_threshold_{t}"):
        mlflow.set_tag("experiment_type", "threshold_analysis")
        mlflow.log_param("threshold",    t)
        mlflow.log_param("base_run_id",  champion_run_id)

        y_pred_t        = (y_proba >= t).astype(int)
        cm_t            = confusion_matrix(y_test, y_pred_t, labels=[0, 1])
        tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
        prec = tp_t / (tp_t + fp_t + 1e-9)
        rec  = tp_t / (tp_t + fn_t + 1e-9)

        mlflow.log_metrics({
            "precision":        round(prec, 4),
            "recall":           round(rec, 4),
            "false_positives":  int(fp_t),
            "fraud_missed":     int(fn_t),
            "fraud_caught_pct": round(rec * 100, 2),
        })

print("All threshold runs logged!")
print("View them at http://localhost:5000")

All threshold runs logged!
View them at http://localhost:5000


In [22]:
runs_df = mlflow.search_runs(
    experiment_names=["fraud-detection"],
    order_by=["metrics.auc_pr DESC"]
)

cols = [
    "tags.mlflow.runName",
    "metrics.auc_pr",
    "metrics.auc_roc",
    "metrics.precision",
    "metrics.recall",
    "metrics.fraud_caught_pct",
]
available = [c for c in cols if c in runs_df.columns]
print(runs_df[available].head(10).to_string(index=False))

         tags.mlflow.runName  metrics.auc_pr  metrics.auc_roc  metrics.precision  metrics.recall  metrics.fraud_caught_pct
        xgboost_optuna_tuned          0.8814           0.9803             0.7727          0.8673                     86.73
baseline_logistic_regression          0.6979           0.9682                NaN             NaN                       NaN
       xgboost_threshold_0.7             NaN              NaN             0.8137          0.8469                     84.69
       xgboost_threshold_0.6             NaN              NaN             0.8077          0.8571                     85.71
       xgboost_threshold_0.5             NaN              NaN             0.7727          0.8673                     86.73
       xgboost_threshold_0.4             NaN              NaN             0.7522          0.8673                     86.73
       xgboost_threshold_0.3             NaN              NaN             0.7273          0.8980                     89.80


In [23]:
best_run = runs_df[runs_df["metrics.auc_pr"].notna()].iloc[0]

print("=" * 50)
print("  WEEK 4 COMPLETE")
print("=" * 50)
print(f"  Experiment:  fraud-detection")
print(f"  Total runs:  {len(runs_df)}")
print(f"  Best run:    {best_run.get('tags.mlflow.runName', '—')}")
print(f"  Best AUC-PR: {best_run['metrics.auc_pr']}")
print(f"\n  Model registered: fraud_detector v{latest.version} (Production)")
print(f"\n  View all runs at: http://localhost:5000")
print("=" * 50)

  WEEK 4 COMPLETE
  Experiment:  fraud-detection
  Total runs:  7
  Best run:    xgboost_optuna_tuned
  Best AUC-PR: 0.8814

  Model registered: fraud_detector v1 (Production)

  View all runs at: http://localhost:5000
